# 01 — Data Preprocessing & Exploration
Load, clean, normalize, and visualize both METR-LA and PEMS-BAY datasets.

**No training here** — just data analysis and preparation.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src import config
from src.config import set_seed
set_seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

## 1. Load Raw Data

In [ ]:
df_la = pd.read_csv(config.METR_LA_PATH, index_col=0, parse_dates=True)
print("=== METR-LA ===")
print(f"Shape: {df_la.shape} (timesteps x sensors)")
print(f"Time: {df_la.index[0]} to {df_la.index[-1]}")
print(f"Interval: {df_la.index[1] - df_la.index[0]}")

In [ ]:
df_bay = pd.read_csv(config.PEMS_BAY_PATH, index_col=0, parse_dates=True)
print("=== PEMS-BAY ===")
print(f"Shape: {df_bay.shape} (timesteps x sensors)")
print(f"Time: {df_bay.index[0]} to {df_bay.index[-1]}")

## 2. Basic Statistics

In [ ]:
for name, df in [('METR-LA', df_la), ('PEMS-BAY', df_bay)]:
    vals = df.values
    print(f"\n=== {name} ===")
    print(f"Missing (NaN): {df.isna().sum().sum()} ({df.isna().mean().mean():.2%})")
    print(f"Zero values:   {(df==0).sum().sum()} ({(df==0).mean().mean():.2%})")
    print(f"Speed range:   [{vals[vals>0].min():.1f}, {vals.max():.1f}] mph")
    print(f"Mean speed:    {vals[vals>0].mean():.1f} mph")

## 3. Time Series Visualization

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
for i, col in enumerate(df_la.columns[:3]):
    axes[0].plot(df_la.index[:2000], df_la[col].iloc[:2000], label=f'Sensor {col}', alpha=0.8)
axes[0].set_title('METR-LA — Traffic Speed (3 sensors, ~7 days)', fontsize=13)
axes[0].set_ylabel('Speed (mph)'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

for i, col in enumerate(df_bay.columns[:3]):
    axes[1].plot(df_bay.index[:2000], df_bay[col].iloc[:2000], label=f'Sensor {col}', alpha=0.8)
axes[1].set_title('PEMS-BAY — Traffic Speed (3 sensors, ~7 days)', fontsize=13)
axes[1].set_ylabel('Speed (mph)'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../results/plots/raw_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Speed Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df_la.values.flatten(), bins=100, color='#3498db', alpha=0.7, edgecolor='white')
axes[0].set_title('METR-LA Speed Distribution', fontsize=13); axes[0].set_xlabel('Speed (mph)')
axes[0].axvline(df_la.values.mean(), color='red', linestyle='--', label=f'Mean={df_la.values.mean():.1f}')
axes[0].legend()

axes[1].hist(df_bay.values.flatten(), bins=100, color='#e67e22', alpha=0.7, edgecolor='white')
axes[1].set_title('PEMS-BAY Speed Distribution', fontsize=13); axes[1].set_xlabel('Speed (mph)')
axes[1].axvline(df_bay.values.mean(), color='red', linestyle='--', label=f'Mean={df_bay.values.mean():.1f}')
axes[1].legend()
plt.tight_layout()
plt.savefig('../results/plots/speed_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Zero Values Analysis (Sensor Failures)

In [ ]:
zero_pct = (df_la == 0).mean() * 100
fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(range(len(zero_pct)), zero_pct.values, color='#e74c3c', alpha=0.7, width=1.0)
ax.set_xlabel('Sensor Index'); ax.set_ylabel('Zero Values (%)')
ax.set_title('METR-LA — Zero Readings per Sensor (sensor failures)', fontsize=13)
ax.axhline(5, color='black', linestyle='--', alpha=0.5, label='5% threshold'); ax.legend()
plt.tight_layout()
plt.savefig('../results/plots/zero_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Preprocessing Pipeline

In [ ]:
from src.data_loader import load_csv, handle_missing_values, normalize_data, create_sequences, split_data

raw, sensor_ids, timestamps = load_csv(config.METR_LA_PATH)
print(f"Before cleaning: zeros={int((raw==0).sum())}, NaN={int(np.isnan(raw).sum())}")

cleaned = handle_missing_values(raw)
print(f"After cleaning:  zeros={int((cleaned==0).sum())}, NaN={int(np.isnan(cleaned).sum())}")
print(f"Speed range: [{cleaned.min():.2f}, {cleaned.max():.2f}]")

In [ ]:
# Z-Score Normalization
train_end = int(len(cleaned) * config.TRAIN_RATIO)
mean_train = cleaned[:train_end].mean(axis=0)
std_train = cleaned[:train_end].std(axis=0); std_train[std_train < 1e-5] = 1.0
normalized = (cleaned - mean_train) / std_train

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(cleaned.flatten()[::100], bins=80, color='#3498db', alpha=0.7)
axes[0].set_title('Before Normalization'); axes[0].set_xlabel('Speed (mph)')
axes[1].hist(normalized.flatten()[::100], bins=80, color='#2ecc71', alpha=0.7)
axes[1].set_title('After Z-Score Normalization'); axes[1].set_xlabel('Normalized')
plt.tight_layout()
plt.savefig('../results/plots/normalization.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Sliding Window Sequences

In [ ]:
X, Y = create_sequences(normalized, config.SEQ_LEN, config.PRED_LEN)
splits = split_data(X, Y, config.TRAIN_RATIO, config.VAL_RATIO)

print(f"Input:  {config.SEQ_LEN} steps = {config.SEQ_LEN*5} min")
print(f"Output: {config.PRED_LEN} steps = {config.PRED_LEN*5} min")
print(f"X: {X.shape}, Y: {Y.shape}")
for name, (x,y) in splits.items():
    print(f"  {name}: {len(x)} samples ({len(x)/len(X)*100:.0f}%)")

In [ ]:
# Visualize train/val/test split
fig, ax = plt.subplots(figsize=(14, 3))
s = cleaned[:, 0]; n = len(s)
t1 = int(n*config.TRAIN_RATIO); t2 = int(n*(config.TRAIN_RATIO+config.VAL_RATIO))
ax.plot(range(t1), s[:t1], color='#3498db', label='Train 70%', alpha=0.8)
ax.plot(range(t1,t2), s[t1:t2], color='#2ecc71', label='Val 10%', alpha=0.8)
ax.plot(range(t2,n), s[t2:], color='#e74c3c', label='Test 20%', alpha=0.8)
ax.set_title('Chronological Train/Val/Test Split', fontsize=13)
ax.set_xlabel('Timestep'); ax.set_ylabel('Speed (mph)'); ax.legend(); ax.grid(True,alpha=0.3)
plt.tight_layout()
plt.savefig('../results/plots/data_split.png', dpi=150, bbox_inches='tight')
plt.show()